In [13]:
#import libraries

import pandas as pd
import sqlite3

In [14]:
#Install the ipython-sql library

!pip install ipython-sql

# Healthcare Utilization & Authorization Analysis (SQL Portfolio Project)

This project analyzes synthetic healthcare authorization and utilization data using SQL inside Jupyter Notebook.

It demonstrates skills in:
- Healthcare data modeling
- SQL joins, aggregations, and TAT calculations
- Identifying utilization trends
- Analyzing approval/denial patterns
- Presenting insights for operational improvement


In [39]:
%load_ext sql
%sql sqlite:///healthcare_analytics.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [40]:
%%sql
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    gender TEXT,
    state TEXT
);


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


[]

In [41]:
%%sql
CREATE TABLE procedures (
    procedure_code TEXT PRIMARY KEY,
    description TEXT,
    category TEXT
);


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


[]

In [42]:
%%sql
CREATE TABLE authorizations (
    auth_id INTEGER PRIMARY KEY,
    member_id INTEGER,
    procedure_code TEXT,
    request_date TEXT,
    decision_date TEXT,
    status TEXT,
    FOREIGN KEY(member_id) REFERENCES members(member_id),
    FOREIGN KEY(procedure_code) REFERENCES procedures(procedure_code)
);


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


[]

In [43]:
%%sql
INSERT INTO members (name, age, gender, state) VALUES
('Alyssa M.', 32, 'F', 'TX'),
('John R.', 45, 'M', 'TX'),
('Maria S.', 29, 'F', 'NM'),
('David P.', 53, 'M', 'OK');

   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
4 rows affected.


[]

In [44]:
%%sql
INSERT INTO procedures (procedure_code, description, category) VALUES
('99213', 'Office Visit – Established Patient', 'Primary Care'),
('97110', 'Physical Therapy – Therapeutic Exercises', 'Rehab'),
('70551', 'MRI Brain Without Contrast', 'Imaging'),
('93000', 'Electrocardiogram (ECG)', 'Cardiology');


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
4 rows affected.


[]

In [45]:
%%sql
INSERT INTO authorizations (member_id, procedure_code, request_date, decision_date, status) VALUES
(1, '70551', '2024-01-02', '2024-01-05', 'Approved'),
(2, '97110', '2024-01-03', '2024-01-10', 'Denied'),
(3, '99213', '2024-01-05', '2024-01-06', 'Approved'),
(1, '93000', '2024-01-07', '2024-01-09', 'Approved'),
(4, '70551', '2024-01-08', '2024-01-20', 'Denied');


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
5 rows affected.


[]

In [46]:
%%sql
SELECT * FROM authorizations;


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


auth_id,member_id,procedure_code,request_date,decision_date,status
1,1,70551,2024-01-02,2024-01-05,Approved
2,2,97110,2024-01-03,2024-01-10,Denied
3,3,99213,2024-01-05,2024-01-06,Approved
4,1,93000,2024-01-07,2024-01-09,Approved
5,4,70551,2024-01-08,2024-01-20,Denied


In [47]:
%%sql
SELECT 
    a.auth_id,
    m.name AS member,
    m.age,
    p.description AS procedure,
    p.category,
    a.status,
    a.request_date,
    a.decision_date
FROM authorizations a
JOIN members m ON a.member_id = m.member_id
JOIN procedures p ON a.procedure_code = p.procedure_code;


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


auth_id,member,age,procedure,category,status,request_date,decision_date
1,Alyssa M.,32,MRI Brain Without Contrast,Imaging,Approved,2024-01-02,2024-01-05
2,John R.,45,Physical Therapy – Therapeutic Exercises,Rehab,Denied,2024-01-03,2024-01-10
3,Maria S.,29,Office Visit – Established Patient,Primary Care,Approved,2024-01-05,2024-01-06
4,Alyssa M.,32,Electrocardiogram (ECG),Cardiology,Approved,2024-01-07,2024-01-09
5,David P.,53,MRI Brain Without Contrast,Imaging,Denied,2024-01-08,2024-01-20


In [48]:
%%sql
SELECT
    a.auth_id,
    m.name,
    p.description,
    a.status,
    julianday(a.decision_date) - julianday(a.request_date) AS tat_days
FROM authorizations a
JOIN members m ON a.member_id = m.member_id
JOIN procedures p ON a.procedure_code = p.procedure_code;


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


auth_id,name,description,status,tat_days
1,Alyssa M.,MRI Brain Without Contrast,Approved,3.0
2,John R.,Physical Therapy – Therapeutic Exercises,Denied,7.0
3,Maria S.,Office Visit – Established Patient,Approved,1.0
4,Alyssa M.,Electrocardiogram (ECG),Approved,2.0
5,David P.,MRI Brain Without Contrast,Denied,12.0


In [49]:
%%sql
SELECT 
    status,
    COUNT(*) AS count
FROM authorizations
GROUP BY status;


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


status,count
Approved,3
Denied,2


In [50]:
%%sql
SELECT 
    p.category,
    COUNT(*) AS request_count
FROM authorizations a
JOIN procedures p ON a.procedure_code = p.procedure_code
GROUP BY p.category
ORDER BY request_count DESC;


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


category,request_count
Imaging,2
Rehab,1
Primary Care,1
Cardiology,1


In [51]:
%%sql
SELECT 
    p.category,
    ROUND(AVG(julianday(a.decision_date) - julianday(a.request_date)), 2) AS avg_tat
FROM authorizations a
JOIN procedures p ON a.procedure_code = p.procedure_code
GROUP BY p.category
ORDER BY avg_tat DESC;


   sqlite:///healthcare.db
 * sqlite:///healthcare_analytics.db
   sqlite:///practice.db
Done.


category,avg_tat
Imaging,7.5
Rehab,7.0
Cardiology,2.0
Primary Care,1.0


## Key Insights

- Imaging (MRI Brain) had the **longest TAT**, suggesting workflow delays.
- Physical Therapy had a **higher denial rate**, which is common due to medical necessity criteria.
- Primary Care visits were approved quickly, showing efficient processing.
- Members in Texas generated the most requests, consistent with regional enrollment.
- Approval rate was strong overall, but denials clustered around specific procedure types.

## What this project demonstrates

- Healthcare data modeling (members, procedures, authorizations)
- SQL joins, aggregations, and TAT calculations
- Ability to analyze UM workflows and identify bottlenecks
- Realistic healthcare analytics aligned with UM/UR operations
- Clean, professional documentation suitable for GitHub
